# Gradient Explosion Prevention: SASSHA + EMA + Hessian Clipping
#
Tests **Hessian Clipping** (NeurIPS 2025) as a defense against gradient explosion
in SASSHA + EMA on Incremental CIFAR-100 with ResNet-18.
#
**Methods**:
#
1. **SASSHA + EMA** — Baseline (no guard, explodes at ~epoch 2200)
2. **SASSHA + EMA + Hessian Clipping** — Clamps `exp_hessian_diag` to `[0, τ]`

## 1. Imports and Setup

In [ ]:
pip install torch==2.1.0 torchvision==0.16.0 --index-url https://download.pytorch.org/whl/cu118

In [ ]:
pip install mlproj-manager==0.0.29

In [ ]:
import os, sys, json, time, pickle, copy, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Optimizer
from torch.utils.data import DataLoader
from torchvision import transforms
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'serif'
matplotlib.rcParams['font.size'] = 11

sys.path.append("/kaggle/input/datasets/bngtbnh04/lop-src")
from lop.nets.torchvision_modified_resnet import build_resnet18, kaiming_init_resnet_module
from lop.incremental_cifar.post_run_analysis import compute_dormant_units_proportion

from mlproj_manager.problems import CifarDataSet
from mlproj_manager.util.data_preprocessing_and_transformations import (
    ToTensor, Normalize, RandomCrop, RandomHorizontalFlip, RandomRotator
)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

results_dir = "anti_overfit_results"
os.makedirs(results_dir, exist_ok=True)

print("✓ Imports done")

## 2. Metrics

In [ ]:
@torch.no_grad()
def compute_avg_weight_magnitude(net):
    n, s = 0, 0.0
    for p in net.parameters():
        n += p.numel()
        s += torch.sum(torch.abs(p)).item()
    return s / n if n > 0 else 0.0

def compute_stable_rank(sv):
    if len(sv) == 0:
        return 0
    sorted_sv = np.flip(np.sort(sv))
    cumsum = np.cumsum(sorted_sv) / np.sum(sv)
    return int(np.sum(cumsum < 0.99) + 1)

def compute_stable_rank_from_activations(act):
    from scipy.linalg import svd
    if act.ndim > 2:
        act = act.reshape(act.shape[0], -1)
    if act.shape[0] == 0 or act.shape[1] == 0:
        return 0
    try:
        sv = svd(act, compute_uv=False, lapack_driver="gesvd")
        return compute_stable_rank(sv)
    except:
        return 0

print("✓ Metrics defined")

## 3. Load CIFAR-100

In [ ]:
mean = (0.5071, 0.4865, 0.4409)
std = (0.2673, 0.2564, 0.2762)

train_transformations = transforms.Compose([
    ToTensor(swap_color_axis=True), Normalize(mean=mean, std=std),
    RandomHorizontalFlip(p=0.5), RandomCrop(size=32, padding=4, padding_mode="reflect"),
    RandomRotator(degrees=(0, 15))
])
eval_transformations = transforms.Compose([
    ToTensor(swap_color_axis=True), Normalize(mean=mean, std=std)
])

data_path = (lambda p: (os.makedirs(p, exist_ok=True), p)[1])("/kaggle/working/incremental_cifar/data")

train_data_full = CifarDataSet(
    root_dir=data_path, train=True, cifar_type=100,
    device=None, image_normalization="max", label_preprocessing="one-hot", use_torch=True)
test_data = CifarDataSet(
    root_dir=data_path, train=False, cifar_type=100,
    device=None, image_normalization="max", label_preprocessing="one-hot", use_torch=True)

def get_train_val_indices(cifar_data, num_classes=100):
    val_idx = torch.zeros(5000, dtype=torch.int32)
    train_idx = torch.zeros(45000, dtype=torch.int32)
    cv, ct = 0, 0
    for i in range(num_classes):
        ci = torch.argwhere(cifar_data.data["labels"][:, i] == 1).flatten()
        val_idx[cv:cv+50] = ci[:50]
        train_idx[ct:ct+450] = ci[50:]
        cv += 50
        ct += 450
    return train_idx, val_idx

train_indices, val_indices = get_train_val_indices(train_data_full)

def subsample_cifar(indices, cifar_data):
    idx = indices.numpy() if isinstance(indices, torch.Tensor) else indices
    cifar_data.data["data"] = cifar_data.data["data"][idx]
    cifar_data.data["labels"] = cifar_data.data["labels"][idx]
    cifar_data.integer_labels = torch.tensor(cifar_data.integer_labels)[idx].tolist()
    cifar_data.current_data = cifar_data.partition_data()

train_data = copy.deepcopy(train_data_full)
val_data = copy.deepcopy(train_data_full)
subsample_cifar(train_indices, train_data)
subsample_cifar(val_indices, val_data)
train_data.set_transformation(train_transformations)
val_data.set_transformation(eval_transformations)
test_data.set_transformation(eval_transformations)

print(f"✓ CIFAR-100: Train={len(train_data.data['data'])}, "
      f"Val={len(val_data.data['data'])}, Test={len(test_data.data['data'])}")

## 4. SASSHA Optimizer
#
Sharpness-Aware Adaptive Second-Order (diagonal Hessian + SAM).
Faithful to: https://github.com/LOG-postech/Sassha (ICML 2025)

In [ ]:
# SASSHA — Sharpness-Aware Adaptive Second-Order Optimization
#      with Stable Hessian Approximation
# Source: https://github.com/LOG-postech/Sassha/blob/master/optimizers/sassha.py
# Paper: Shin et al., ICML 2025 (arXiv:2502.18153)
#
# Key differences from Shampoo:
#   - Uses DIAGONAL Hessian (Hutchinson trace) not Kronecker factors
#   - Includes SAM perturbation (sharpness-aware ascent step)
#   - Update rule: Adam-like with Hessian diagonal as second moment
#   - lazy_hessian: recompute Hessian every N steps
#   - hessian_power: controls curvature strength (scheduled)

class SASSHA(Optimizer):
    """SASSHA optimizer — faithful to official LOG-postech implementation.

    Training loop (caller must follow this protocol):
        loss = loss_fn(model(input))
        loss.backward()
        optimizer.perturb_weights(zero_grad=True)
        loss_fn(model(input)).backward(create_graph=True)
        optimizer.unperturb()
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
    """

    def __init__(self, params, lr=0.15, betas=(0.9, 0.999), weight_decay=0.0,
                 rho=0.2, lazy_hessian=10, n_samples=1,
                 perturb_eps=1e-12, eps=1e-4, hessian_power=0.5,
                 adaptive=False, seed=0):
        defaults = dict(lr=lr, betas=betas, weight_decay=weight_decay,
                        rho=rho, perturb_eps=perturb_eps, eps=eps)
        super().__init__(params, defaults)

        self.lazy_hessian = lazy_hessian
        self.n_samples = n_samples
        self.adaptive = adaptive
        self.seed = seed
        self.hessian_power_t = hessian_power

        for p in self._get_params():
            p.hess = 0.0
            self.state[p]["hessian step"] = 0

        self.generator = torch.Generator().manual_seed(self.seed)

    def _get_params(self):
        return (p for group in self.param_groups
                for p in group['params'] if p.requires_grad)

    def zero_hessian(self):
        for p in self._get_params():
            if (not isinstance(p.hess, float)
                    and self.state[p]["hessian step"] % self.lazy_hessian == 0):
                p.hess.zero_()

    @torch.no_grad()
    def set_hessian(self):
        """Hutchinson trace estimator: Hessian diagonal ≈ E[z ⊙ (Hz)]."""
        params = []
        for p in filter(lambda p: p.grad is not None, self._get_params()):
            if self.state[p]["hessian step"] % self.lazy_hessian == 0:
                params.append(p)
            self.state[p]["hessian step"] += 1

        if len(params) == 0:
            return

        if self.generator.device != params[0].device:
            self.generator = torch.Generator(params[0].device).manual_seed(self.seed)

        grads = [p.grad for p in params]

        last_sample = self.n_samples - 1
        for i in range(self.n_samples):
            zs = [torch.randint(0, 2, p.size(), generator=self.generator,
                                device=p.device) * 2.0 - 1.0 for p in params]
            h_zs = torch.autograd.grad(
                grads, params, grad_outputs=zs,
                only_inputs=True, retain_graph=i < last_sample)
            for h_z, z, p in zip(h_zs, zs, params):
                p.hess += h_z * z / self.n_samples

    @torch.no_grad()
    def perturb_weights(self, zero_grad=True):
        """SAM ascent step: w → w + ρ · ∇L / ‖∇L‖."""
        grad_norm = self._grad_norm(weight_adaptive=self.adaptive)
        for group in self.param_groups:
            scale = group["rho"] / (grad_norm + group["perturb_eps"])
            for p in group["params"]:
                if p.grad is None:
                    continue
                e_w = p.grad * scale.to(p)
                if self.adaptive:
                    e_w *= torch.pow(p, 2)
                p.add_(e_w)
                self.state[p]['e_w'] = e_w

        if zero_grad:
            self.zero_grad()

    @torch.no_grad()
    def unperturb(self):
        """Restore weights: w + e(w) → w."""
        for group in self.param_groups:
            for p in group['params']:
                if 'e_w' in self.state[p]:
                    p.data.sub_(self.state[p]['e_w'])

    @torch.no_grad()
    def _grad_norm(self, weight_adaptive=False):
        norm = torch.norm(
            torch.stack([
                ((torch.abs(p.data) if weight_adaptive else 1.0) * p.grad).norm(p=2)
                for group in self.param_groups for p in group["params"]
                if p.grad is not None
            ]),
            p=2
        )
        return norm

    @torch.no_grad()
    def step(self, closure=None, compute_hessian=True):
        loss = None
        if closure is not None:
            loss = closure()

        if compute_hessian:
            self.zero_hessian()
            self.set_hessian()

        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None or p.hess is None:
                    continue

                if isinstance(p.hess, (int, float)):
                    p.hess = torch.zeros_like(p.data)
                else:
                    p.hess = p.hess.abs().clone()

                p.mul_(1 - group['lr'] * group['weight_decay'])

                state = self.state[p]
                if len(state) == 2:
                    state['step'] = 0
                    state['exp_avg'] = torch.zeros_like(p.data)
                    state['exp_hessian_diag'] = torch.zeros_like(p.data)
                    state['bias_correction2'] = 0

                exp_avg = state['exp_avg']
                exp_hessian_diag = state['exp_hessian_diag']
                beta1, beta2 = group['betas']
                state['step'] += 1

                exp_avg.mul_(beta1).add_(p.grad, alpha=1 - beta1)
                bias_correction1 = 1 - beta1 ** state['step']

                if (state['hessian step'] - 1) % self.lazy_hessian == 0:
                    exp_hessian_diag.mul_(beta2).add_(p.hess, alpha=1 - beta2)
                    bias_correction2 = 1 - beta2 ** state['step']
                    state['bias_correction2'] = bias_correction2 ** self.hessian_power_t

                step_size = group['lr'] / bias_correction1
                denom = ((exp_hessian_diag ** self.hessian_power_t)
                         / state['bias_correction2']).add_(group['eps'])
                p.addcdiv_(exp_avg, denom, value=-step_size)

        return loss


def _disable_running_stats(model):
    """Prevent BatchNorm from updating running stats during SAM perturbation pass."""
    def _disable(module):
        if isinstance(module, nn.BatchNorm2d):
            module.backup_momentum = module.momentum
            module.momentum = 0
    model.apply(_disable)

def _enable_running_stats(model):
    """Restore BatchNorm momentum after SAM perturbation pass."""
    def _enable(module):
        if isinstance(module, nn.BatchNorm2d) and hasattr(module, 'backup_momentum'):
            module.momentum = module.backup_momentum
    model.apply(_enable)


print("✓ SASSHA optimizer defined")

## 5. EMA Wrapper
#
Exponential Moving Average of weights — used for evaluation only.

In [ ]:
class EMAWrapper:
    """Exponential Moving Average of model weights.
    Used ONLY for evaluation; training runs on original weights."""

    def __init__(self, model, decay=0.999):
        self.decay = decay
        self._shadow = {n: p.data.clone() for n, p in model.named_parameters()}
        self._backup = {}

    @torch.no_grad()
    def update(self, model):
        for n, p in model.named_parameters():
            self._shadow[n].mul_(self.decay).add_(p.data, alpha=1 - self.decay)

    @torch.no_grad()
    def apply(self, model):
        self._backup = {n: p.data.clone() for n, p in model.named_parameters()}
        for n, p in model.named_parameters():
            p.data.copy_(self._shadow[n])

    @torch.no_grad()
    def restore(self, model):
        for n, p in model.named_parameters():
            if n in self._backup:
                p.data.copy_(self._backup[n])
        self._backup.clear()

    @torch.no_grad()
    def reset(self, model):
        self._shadow = {n: p.data.clone() for n, p in model.named_parameters()}


print("✓ EMA wrapper defined")

## 6a. Gradient Explosion Guard (6 methods from top-tier venues)
#
**Problem**: SASSHA + EMA suffers gradient explosion at epoch ~2200+ due to stale Hessian,
SAM perturbation amplification, and no gradient clipping.
#
**Methods integrated** (all from NeurIPS / ICML / ICLR 2021-2025):
1. **AGC** — Adaptive Gradient Clipping (NFNet, ICML 2021)
2. **ZClip** — Z-score anomaly detection for gradient spikes (2025)
3. **Hessian Clipping** — Clip Hessian diagonal (NeurIPS 2025)
4. **SPAM-style Momentum Reset** — Reset corrupted buffers on spike (ICLR 2025)
5. **NaN-Guard Checkpoint Rollback** — Auto-recover from NaN/explosion
6. **Adaptive ρ Scheduling** — Reduce SAM perturbation at task boundaries (JMLR 2024)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# GradientExplosionGuard — unified defense combining 6 techniques
# ═══════════════════════════════════════════════════════════════════════

class GradientExplosionGuard:
    """Unified gradient explosion prevention combining 6 top-tier methods.

    [1] AGC — Brock et al., ICML 2021 (NFNet)
    [2] ZClip — z-score spike detection, arXiv 2025
    [3] Hessian Clipping — Sadiev et al., NeurIPS 2025
    [4] SPAM momentum reset — Huang et al., ICLR 2025
    [5] NaN-guard + checkpoint rollback (PaLM / Llama practice)
    [6] Adaptive rho scheduling (JMLR 2024 SAM dynamics)
    """

    def __init__(
        self,
        agc_clip_factor=0.01,           # [1] max ‖∇W‖/‖W‖ ratio per param
        zclip_zscore_thresh=3.0,        # [2] z-score threshold for spike detection
        zclip_ema_decay=0.99,           # [2] EMA decay for gradient norm tracking
        hessian_clip_value=1e3,         # [3] max absolute value for Hessian diag
        spike_loss_factor=10.0,         # [4][5] loss spike = current > factor * EMA
        max_grad_norm=1.0,              # global grad norm clip (fallback safety)
        enable_agc=True,
        enable_zclip=True,
        enable_hessian_clip=True,
        enable_momentum_reset=True,
        enable_nan_guard=True,
    ):
        self.agc_clip_factor = agc_clip_factor
        self.zclip_zscore_thresh = zclip_zscore_thresh
        self.zclip_ema_decay = zclip_ema_decay
        self.hessian_clip_value = hessian_clip_value
        self.spike_loss_factor = spike_loss_factor
        self.max_grad_norm = max_grad_norm

        self.enable_agc = enable_agc
        self.enable_zclip = enable_zclip
        self.enable_hessian_clip = enable_hessian_clip
        self.enable_momentum_reset = enable_momentum_reset
        self.enable_nan_guard = enable_nan_guard

        # ZClip state: EMA of gradient norm and variance
        self._gnorm_ema = None
        self._gnorm_var_ema = None
        self._loss_ema = None

        # Counters
        self.spike_count = 0
        self.nan_count = 0
        self.agc_clip_count = 0
        self.hessian_clip_count = 0

    # ─── [1] Adaptive Gradient Clipping (AGC, NFNet) ───
    @torch.no_grad()
    def apply_agc(self, model):
        """Per-parameter clipping based on ‖∇W‖/‖W‖ ratio."""
        if not self.enable_agc:
            return
        clipped = 0
        for p in model.parameters():
            if p.grad is None:
                continue
            p_norm = p.data.norm(2).clamp(min=1e-6)
            g_norm = p.grad.data.norm(2)
            max_g_norm = p_norm * self.agc_clip_factor
            if g_norm > max_g_norm:
                p.grad.data.mul_(max_g_norm / g_norm)
                clipped += 1
        self.agc_clip_count += clipped

    # ─── [2] ZClip: z-score based spike detection ───
    def detect_spike_zclip(self, model):
        """Returns True if current gradient norm is a statistical anomaly."""
        if not self.enable_zclip:
            return False
        gnorm = torch.nn.utils.clip_grad_norm_(model.parameters(), float('inf'))
        gnorm = gnorm.item() if isinstance(gnorm, torch.Tensor) else gnorm

        if self._gnorm_ema is None:
            self._gnorm_ema = gnorm
            self._gnorm_var_ema = 0.0
            return False

        d = self.zclip_ema_decay
        self._gnorm_var_ema = d * self._gnorm_var_ema + (1 - d) * (gnorm - self._gnorm_ema) ** 2
        self._gnorm_ema = d * self._gnorm_ema + (1 - d) * gnorm

        std = max(math.sqrt(self._gnorm_var_ema), 1e-8)
        zscore = (gnorm - self._gnorm_ema) / std
        is_spike = zscore > self.zclip_zscore_thresh
        if is_spike:
            self.spike_count += 1
            clip_val = self._gnorm_ema + self.zclip_zscore_thresh * std
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip_val)
        return is_spike

    # ─── [3] Hessian Clipping (NeurIPS 2025) ───
    @torch.no_grad()
    def clip_hessian(self, optimizer):
        """Clip exp_hessian_diag to prevent extreme curvature values."""
        if not self.enable_hessian_clip:
            return
        tau = self.hessian_clip_value
        clipped = False
        for group in optimizer.param_groups:
            for p in group['params']:
                state = optimizer.state[p]
                if 'exp_hessian_diag' in state:
                    h = state['exp_hessian_diag']
                    if h.max().item() > tau:
                        clipped = True
                    h.clamp_(0, tau)
                if hasattr(p, 'hess') and not isinstance(p.hess, float):
                    p.hess.clamp_(-tau, tau)
        if clipped:
            self.hessian_clip_count += 1

    # ─── [4] SPAM-style momentum reset (ICLR 2025) ───
    @torch.no_grad()
    def maybe_reset_momentum(self, optimizer, force=False):
        """Reset optimizer momentum buffers if spike was detected or forced (task boundary)."""
        if not self.enable_momentum_reset and not force:
            return False
        for group in optimizer.param_groups:
            for p in group['params']:
                state = optimizer.state[p]
                if 'exp_avg' in state:
                    state['exp_avg'].zero_()
                if 'exp_hessian_diag' in state:
                    state['exp_hessian_diag'].zero_()
                if 'bias_correction2' in state:
                    state['bias_correction2'] = 0
                if 'step' in state:
                    state['step'] = 0
                if 'hessian step' in state:
                    state['hessian step'] = 0
                if hasattr(p, 'hess'):
                    p.hess = 0.0
        return True

    # ─── [5] NaN-guard: detect NaN/explosion in loss ───
    def check_loss_health(self, loss_value):
        """Returns ('ok', None) or ('nan', reason) or ('spike', reason)."""
        if not self.enable_nan_guard:
            return 'ok', None
        if not math.isfinite(loss_value):
            self.nan_count += 1
            return 'nan', f'loss={loss_value}'
        if self._loss_ema is None:
            self._loss_ema = loss_value
            return 'ok', None
        self._loss_ema = 0.95 * self._loss_ema + 0.05 * min(loss_value, self._loss_ema * 5)
        if loss_value > self.spike_loss_factor * max(self._loss_ema, 0.1):
            return 'spike', f'loss={loss_value:.2f} >> EMA={self._loss_ema:.2f}'
        return 'ok', None

    # ─── [6] Adaptive rho scheduling ───
    @staticmethod
    def compute_adaptive_rho(base_rho, epoch_in_task, warmup_epochs=20):
        """Linearly warm up SAM perturbation radius from 0 to base_rho
        over the first warmup_epochs of each task to avoid amplifying
        instability when the loss landscape has just shifted."""
        if epoch_in_task >= warmup_epochs:
            return base_rho
        return base_rho * (epoch_in_task / warmup_epochs)

    # ─── Global fallback: standard gradient norm clipping ───
    @torch.no_grad()
    def clip_global_grad_norm(self, model):
        torch.nn.utils.clip_grad_norm_(model.parameters(), self.max_grad_norm)

    def reset_loss_tracking(self):
        """Call at task boundaries to reset loss EMA."""
        self._loss_ema = None

    def summary(self):
        return (f"Guard stats: spikes={self.spike_count}, "
                f"nan={self.nan_count}, agc_clips={self.agc_clip_count}, "
                f"hess_clips={self.hessian_clip_count}")


print("✓ GradientExplosionGuard defined (6 methods)")

(removed)

In [ ]:
# (removed — not used in current experiment)

## 6. Configs
#
SASSHA + EMA baseline vs SASSHA + EMA + Hessian Clipping.

In [ ]:
NUM_CLASSES = 100
SEED = 42
CKPT_EVERY = 50  # save checkpoint to disk every N epochs

_SHARED = dict(
    num_epochs=4000, batch_size=90, class_increase_frequency=200,
    use_early_stopping=True,
    use_ema=False, ema_decay=0.999,
)

_SASSHA_BASE = dict(
    optimizer='sassha', batch_size=90,
    lr=0.15, betas=(0.9, 0.999), weight_decay=1e-4,
    rho=0.2, lazy_hessian=10, n_samples=1,
    eps=1e-4, hessian_power=0.5,
    lr_milestones=[60, 120, 160], lr_gamma=0.2,
)

CONFIGS = {
    # ─── SASSHA + EMA (baseline, no guard — sẽ nổ ở ~E2200) ───
    'sassha_ema': {
        **_SHARED, **_SASSHA_BASE,
        'use_ema': True, 'ema_decay': 0.999,
    },

    # ─── SASSHA + EMA + Hessian Clipping (NeurIPS 2025) ───
    'sassha_ema_hessclip': {
        **_SHARED, **_SASSHA_BASE,
        'use_ema': True, 'ema_decay': 0.999,
        'use_guard': True,
        'hessian_clip': 1e3,
        'max_grad_norm': float('inf'),
        'rho_warmup_epochs': 0,
        'guard_enable_agc': False,
        'guard_enable_zclip': False,
        'guard_enable_hessian_clip': True,
        'guard_enable_momentum_reset': False,
        'guard_enable_nan_guard': False,
    },
}

METHODS_TO_RUN = [
    'sassha_ema',             # baseline — sẽ nổ ở ~E2200
    'sassha_ema_hessclip',    # + Hessian Clipping only
]

print(f"✓ Will run: {METHODS_TO_RUN}")

## 7. Build Optimizer Helper

In [ ]:
def build_optimizer(config, model):
    return SASSHA(
        model.parameters(),
        lr=config['lr'],
        betas=config.get('betas', (0.9, 0.999)),
        weight_decay=config.get('weight_decay', 1e-5),
        rho=config.get('rho', 0.2),
        lazy_hessian=config.get('lazy_hessian', 10),
        n_samples=config.get('n_samples', 1),
        eps=config.get('eps', 1e-4),
        hessian_power=config.get('hessian_power', 0.5),
        seed=SEED,
    )


print("✓ build_optimizer defined")

## 8. Training Loop

In [ ]:
def _num_classes_at_epoch(epoch, freq, initial=5, step=5, max_cls=100):
    """Derive current_num_classes BEFORE boundary logic at `epoch` runs.

    Boundaries fire at freq, 2*freq, ... (only when epoch > 0).
    Returns the class count as it was BEFORE any boundary at `epoch` fires.
    """
    if epoch <= 0:
        return initial
    n_fired = (epoch - 1) // freq
    return min(initial + n_fired * step, max_cls)


def _ckpt_path(method_name):
    return os.path.join(results_dir, f"ckpt_{method_name}.pt")


def run_method(method_name, config):
    print(f"\n{'='*70}")
    print(f"  Running: {method_name}")
    print(f"  Optimizer: {config['optimizer']}  lr={config['lr']}")
    print(f"  EMA={config['use_ema']}")
    use_guard = config.get('use_guard', False)
    if use_guard:
        active = []
        if config.get('guard_enable_agc', True): active.append('AGC')
        if config.get('guard_enable_zclip', True): active.append('ZClip')
        if config.get('guard_enable_hessian_clip', True): active.append('HessClip')
        if config.get('guard_enable_momentum_reset', True): active.append('SPAMReset')
        if config.get('guard_enable_nan_guard', True): active.append('NaNGuard')
        if config.get('rho_warmup_epochs', 0) > 0: active.append('RhoWarmup')
        print(f"  Guard: {'+'.join(active) if active else '(none)'}")
    print(f"{'='*70}")

    torch.manual_seed(SEED)
    torch.cuda.manual_seed(SEED)
    np.random.seed(SEED)
    all_classes = np.random.RandomState(SEED).permutation(NUM_CLASSES)

    freq = config['class_increase_frequency']

    net = build_resnet18(num_classes=NUM_CLASSES, norm_layer=nn.BatchNorm2d)
    net.apply(kaiming_init_resnet_module)
    net.to(device)

    optimizer = build_optimizer(config, net)

    # ─── Gradient Explosion Guard (per-mechanism toggles from config) ───
    guard = None
    if use_guard:
        guard = GradientExplosionGuard(
            agc_clip_factor=config.get('agc_clip_factor', 0.01),
            zclip_zscore_thresh=config.get('zclip_zscore_thresh', 3.0),
            hessian_clip_value=config.get('hessian_clip', 1e3),
            spike_loss_factor=config.get('spike_loss_factor', 10.0),
            max_grad_norm=config.get('max_grad_norm', 1.0),
            enable_agc=config.get('guard_enable_agc', True),
            enable_zclip=config.get('guard_enable_zclip', True),
            enable_hessian_clip=config.get('guard_enable_hessian_clip', True),
            enable_momentum_reset=config.get('guard_enable_momentum_reset', True),
            enable_nan_guard=config.get('guard_enable_nan_guard', True),
        )

    base_rho = config.get('rho', 0.2)
    rho_warmup_epochs = config.get('rho_warmup_epochs', 0)

    ema = (EMAWrapper(net, config.get('ema_decay', 0.999))
           if config['use_ema'] else None)

    loss_fn = nn.CrossEntropyLoss()

    metrics = {k: [] for k in [
        'train_loss', 'train_acc', 'val_acc', 'test_acc',
        'dormant_after', 'dormant_before', 'stable_rank',
        'avg_weight_mag', 'epoch_time', 'overfit_gap',
    ]}

    # ─── Resume from disk checkpoint if available ───
    start_epoch = 0
    ckpt_file = _ckpt_path(method_name)
    if os.path.isfile(ckpt_file):
        ckpt = torch.load(ckpt_file, map_location=device, weights_only=False)
        net.load_state_dict(ckpt['model'])
        optimizer.load_state_dict(ckpt['optimizer'])
        if ema is not None and 'ema_shadow' in ckpt:
            ema._shadow = ckpt['ema_shadow']
        if guard is not None and 'guard_state' in ckpt:
            gs = ckpt['guard_state']
            guard._loss_ema = gs.get('_loss_ema', 0.0)
            guard._gnorm_ema = gs.get('_gnorm_ema', 0.0)
            guard._gnorm_var_ema = gs.get('_gnorm_var_ema', 0.0)
            guard.spike_count = gs.get('spike_count', 0)
            guard.nan_count = gs.get('nan_count', 0)
            guard.hessian_clip_count = gs.get('hessian_clip_count', 0)
        metrics = ckpt['metrics']
        start_epoch = ckpt['epoch'] + 1
        if 'current_num_classes' in ckpt:
            _saved_num_classes = ckpt['current_num_classes']
        else:
            _saved_num_classes = None
        # After load_state_dict, p.hess (ad-hoc param attribute) is NOT restored.
        # Reset hessian step counters so set_hessian() recomputes on the first step.
        for p in optimizer._get_params():
            p.hess = 0.0
            optimizer.state[p]["hessian step"] = 0
        print(f"  ✓ Resumed from checkpoint epoch {ckpt['epoch']}  ({ckpt_file})")
        del ckpt
        torch.cuda.empty_cache()
    else:
        _saved_num_classes = None
        print(f"  (no checkpoint found, training from scratch)")

    if _saved_num_classes is not None:
        current_num_classes = _saved_num_classes
    else:
        current_num_classes = _num_classes_at_epoch(start_epoch, freq)

    # Data loaders
    train_exp = copy.deepcopy(train_data)
    val_exp = copy.deepcopy(val_data)
    test_exp = copy.deepcopy(test_data)
    train_exp.select_new_partition(all_classes[:current_num_classes])
    val_exp.select_new_partition(all_classes[:current_num_classes])
    test_exp.select_new_partition(all_classes[:current_num_classes])
    train_loader = DataLoader(train_exp, batch_size=config['batch_size'],
                              shuffle=True, num_workers=0)
    val_loader = DataLoader(val_exp, batch_size=50, shuffle=False, num_workers=0)
    test_loader = DataLoader(test_exp, batch_size=100, shuffle=False, num_workers=0)

    dormant_next = copy.deepcopy(train_data_full)
    dormant_next.set_transformation(eval_transformations)
    dormant_prev = copy.deepcopy(train_data_full)
    dormant_prev.set_transformation(eval_transformations)
    ns = current_num_classes
    ne = min(ns + 5, NUM_CLASSES)
    if ns < NUM_CLASSES:
        dormant_next.select_new_partition(all_classes[ns:ne])
        dorm_next_loader = DataLoader(dormant_next, batch_size=1000,
                                      shuffle=True, num_workers=0)
    else:
        dorm_next_loader = None
    dorm_prev_loader = None
    if current_num_classes > 5:
        dormant_prev.select_new_partition(all_classes[:current_num_classes])
        dorm_prev_loader = DataLoader(dormant_prev, batch_size=1000,
                                      shuffle=True, num_workers=0)

    best_val_acc = 0.0
    best_model_state = None
    safe_checkpoint = None  # [5] NaN-guard rollback checkpoint (in-memory)

    for epoch in range(start_epoch, config['num_epochs']):
        t0 = time.time()
        epoch_in_task = epoch % freq

        # ── Task boundary ──
        if epoch > 0 and epoch % freq == 0 and current_num_classes < NUM_CLASSES:
            if config['use_early_stopping'] and best_model_state is not None:
                net.load_state_dict(best_model_state)
                print(f"  → Early stop: loaded best val (acc={best_val_acc:.4f})")
            best_val_acc = 0.0
            best_model_state = None

            if ema is not None:
                ema.reset(net)

            # [4] SPAM-style: reset optimizer state at task boundary (ICLR 2025)
            if guard is not None:
                if guard.enable_momentum_reset:
                    guard.maybe_reset_momentum(optimizer, force=True)
                    print(f"  → Guard: momentum reset at task boundary")
                guard.reset_loss_tracking()

            current_num_classes = min(current_num_classes + 5, NUM_CLASSES)
            train_exp.select_new_partition(all_classes[:current_num_classes])
            val_exp.select_new_partition(all_classes[:current_num_classes])
            test_exp.select_new_partition(all_classes[:current_num_classes])
            train_loader = DataLoader(train_exp, batch_size=config['batch_size'],
                                      shuffle=True, num_workers=0)
            val_loader = DataLoader(val_exp, batch_size=50, shuffle=False, num_workers=0)
            test_loader = DataLoader(test_exp, batch_size=100, shuffle=False, num_workers=0)

            ns = current_num_classes
            ne = min(ns + 5, NUM_CLASSES)
            if ns < NUM_CLASSES:
                dormant_next.select_new_partition(all_classes[ns:ne])
                dorm_next_loader = DataLoader(dormant_next, batch_size=1000,
                                              shuffle=True, num_workers=0)
            else:
                dorm_next_loader = None
            dormant_prev.select_new_partition(all_classes[:current_num_classes])
            dorm_prev_loader = DataLoader(dormant_prev, batch_size=1000,
                                          shuffle=True, num_workers=0)
            print(f"  Task boundary: epoch {epoch} → {current_num_classes} classes")

        # ── Per-task multi-step LR decay (paper: ×gamma at milestones) ──
        milestones = config.get('lr_milestones', [])
        if milestones:
            gamma = config.get('lr_gamma', 0.1)
            base_lr = config['lr']
            decay = gamma ** sum(epoch_in_task >= m for m in milestones)
            new_lr = base_lr * decay
            for g in optimizer.param_groups:
                g['lr'] = new_lr

        # ── Train ──
        net.train()
        rl, ra, nb = 0.0, 0.0, 0

        for sample in train_loader:
            img = sample["image"].to(device)
            lbl = sample["label"].to(device)
            tgt = lbl.argmax(1) if lbl.dim() > 1 and lbl.shape[1] > 1 else lbl

            # ---- SASSHA two-pass protocol ----
            _enable_running_stats(net)
            pred = net(img)[:, all_classes[:current_num_classes]]
            loss = loss_fn(pred, tgt)

            # [5] NaN-guard: check loss before backward
            if guard is not None:
                status, reason = guard.check_loss_health(loss.item())
                if status == 'nan':
                    print(f"  ⚠ NaN loss detected ({reason}), rolling back...")
                    if safe_checkpoint is not None:
                        net.load_state_dict(safe_checkpoint['model'])
                    guard.maybe_reset_momentum(optimizer, force=True)
                    if ema is not None:
                        ema.reset(net)
                    optimizer.zero_grad(set_to_none=True)
                    continue
                elif status == 'spike':
                    print(f"  ⚠ Loss spike ({reason}), clipping + resetting momentum")
                    guard.maybe_reset_momentum(optimizer, force=True)

            loss.backward()

            # [1] AGC — clip gradients per-param (ICML 2021)
            if guard is not None:
                guard.apply_agc(net)

            # [2] ZClip — detect anomalous gradient spikes (2025)
            if guard is not None:
                guard.detect_spike_zclip(net)

            # [6] Adaptive rho warmup at task boundaries (JMLR 2024)
            if guard is not None and rho_warmup_epochs > 0:
                current_rho = GradientExplosionGuard.compute_adaptive_rho(
                    base_rho, epoch_in_task, rho_warmup_epochs)
                for g in optimizer.param_groups:
                    g['rho'] = current_rho

            optimizer.perturb_weights(zero_grad=True)
            _disable_running_stats(net)
            pred_pert = net(img)[:, all_classes[:current_num_classes]]
            loss_pert = loss_fn(pred_pert, tgt)
            loss_pert.backward(create_graph=True)
            optimizer.unperturb()

            # Global fallback gradient norm clip
            if guard is not None:
                guard.clip_global_grad_norm(net)

            # [3] Hessian clipping (NeurIPS 2025):
            # Compute hessian externally → clip → then step with pre-clipped values
            if guard is not None and guard.enable_hessian_clip:
                optimizer.zero_hessian()
                optimizer.set_hessian()
                guard.clip_hessian(optimizer)
                optimizer.step(compute_hessian=False)
            else:
                optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            _enable_running_stats(net)
            rl += loss.item()

            if ema is not None:
                ema.update(net)

            with torch.no_grad():
                pred_eval = net(img)[:, all_classes[:current_num_classes]]
                acc = (pred_eval.argmax(1) == tgt).float().mean()
            ra += acc.item()
            nb += 1

        avg_loss = rl / nb if nb > 0 else float('nan')
        metrics['train_loss'].append(avg_loss)
        metrics['train_acc'].append(ra / nb if nb > 0 else 0)

        # [5] Save safe checkpoint periodically for NaN rollback (in-memory)
        if guard is not None and math.isfinite(avg_loss) and epoch % 10 == 0:
            safe_checkpoint = {'model': copy.deepcopy(net.state_dict()), 'epoch': epoch}

        # ── Eval (use EMA if available) ──
        if ema is not None:
            ema.apply(net)

        net.eval()
        with torch.no_grad():
            va, vb = 0, 0
            for s in val_loader:
                p = net(s["image"].to(device))[:, all_classes[:current_num_classes]]
                t = s["label"].to(device)
                t = t.argmax(1) if t.dim() > 1 and t.shape[1] > 1 else t
                va += (p.argmax(1) == t).float().mean().item()
                vb += 1
            val_acc_epoch = va / vb if vb else 0
            metrics['val_acc'].append(val_acc_epoch)

            ta, tb = 0, 0
            for s in test_loader:
                p = net(s["image"].to(device))[:, all_classes[:current_num_classes]]
                t = s["label"].to(device)
                t = t.argmax(1) if t.dim() > 1 and t.shape[1] > 1 else t
                ta += (p.argmax(1) == t).float().mean().item()
                tb += 1
            metrics['test_acc'].append(ta / tb if tb else 0)

        if val_acc_epoch > best_val_acc:
            best_val_acc = val_acc_epoch
            best_model_state = copy.deepcopy(net.state_dict())

        # ── Restore training weights BEFORE measuring training-model metrics ──
        if ema is not None:
            ema.restore(net)

        # ── Dormant measurement (always on training model) ──
        bn_bk = {}
        for nm, m in net.named_modules():
            if isinstance(m, nn.BatchNorm2d):
                bn_bk[nm] = {'rm': m.running_mean.clone(), 'rv': m.running_var.clone(),
                             'nt': m.num_batches_tracked.clone()}

        net.train()
        with torch.no_grad():
            if dorm_next_loader is not None:
                da, last_act = compute_dormant_units_proportion(
                    net, dorm_next_loader, 0.01)
                sr = compute_stable_rank_from_activations(last_act)
            else:
                da, sr = float('nan'), float('nan')
            for nm, m in net.named_modules():
                if isinstance(m, nn.BatchNorm2d) and nm in bn_bk:
                    m.running_mean.copy_(bn_bk[nm]['rm'])
                    m.running_var.copy_(bn_bk[nm]['rv'])
                    m.num_batches_tracked.copy_(bn_bk[nm]['nt'])
            if dorm_prev_loader is not None:
                db = compute_dormant_units_proportion(
                    net, dorm_prev_loader, 0.01)[0]
            else:
                db = float('nan')

        for nm, m in net.named_modules():
            if isinstance(m, nn.BatchNorm2d) and nm in bn_bk:
                m.running_mean.copy_(bn_bk[nm]['rm'])
                m.running_var.copy_(bn_bk[nm]['rv'])
                m.num_batches_tracked.copy_(bn_bk[nm]['nt'])

        metrics['dormant_after'].append(da)
        metrics['dormant_before'].append(db)
        metrics['stable_rank'].append(sr)
        wm = compute_avg_weight_magnitude(net)
        metrics['avg_weight_mag'].append(wm)
        gap = metrics['train_acc'][-1] - metrics['test_acc'][-1]
        metrics['overfit_gap'].append(gap)

        net.train()

        et = time.time() - t0
        metrics['epoch_time'].append(et)

        if epoch % 50 == 0 or epoch == config['num_epochs'] - 1:
            guard_str = ""
            if guard is not None and epoch % 200 == 0:
                guard_str = f" | {guard.summary()}"
            print(f"  [{method_name}] E{epoch:4d} | Loss={avg_loss:.4f} "
                  f"TestAcc={metrics['test_acc'][-1]:.4f} "
                  f"Gap={gap:.4f} Dorm={da:.4f}/{db:.4f} "
                  f"AvgW={wm:.4f} {et:.1f}s{guard_str}")

        # ── Save disk checkpoint for resume ──
        if epoch % CKPT_EVERY == 0 or epoch == config['num_epochs'] - 1:
            ckpt_data = {
                'epoch': epoch,
                'model': net.state_dict(),
                'optimizer': optimizer.state_dict(),
                'metrics': metrics,
                'current_num_classes': current_num_classes,
            }
            if ema is not None:
                ckpt_data['ema_shadow'] = ema._shadow
            if guard is not None:
                ckpt_data['guard_state'] = {
                    '_loss_ema': guard._loss_ema,
                    '_gnorm_ema': guard._gnorm_ema,
                    '_gnorm_var_ema': guard._gnorm_var_ema,
                    'spike_count': guard.spike_count,
                    'nan_count': guard.nan_count,
                    'hessian_clip_count': guard.hessian_clip_count,
                }
            torch.save(ckpt_data, _ckpt_path(method_name))

    if guard is not None:
        print(f"\n  Final {guard.summary()}")

    return metrics


print("✓ Training loop defined")

## 9. Run All Methods

In [ ]:
all_results = {}
for method in METHODS_TO_RUN:
    all_results[method] = run_method(method, CONFIGS[method])
    with open(os.path.join(results_dir, f"anti_overfit_{method}.pkl"), 'wb') as f:
        pickle.dump(all_results[method], f)
    print(f"  ✓ {method} saved.")

with open(os.path.join(results_dir, "anti_overfit_all_results.pkl"), 'wb') as f:
    pickle.dump(all_results, f)
print(f"\n✓ All results saved.")

## 10. Visualization

In [ ]:
METHOD_STYLES = {
    'sassha_ema':           {'color': '#ff7f0e', 'ls': '-',  'lw': 2.2,
                             'label': 'SASSHA + EMA (baseline)'},
    'sassha_ema_hessclip':  {'color': '#2ca02c', 'ls': '-',  'lw': 2.5,
                             'label': 'SASSHA + EMA + HessClip'},
}

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('SASSHA + EMA: Hessian Clipping vs Baseline — Incremental CIFAR-100',
             fontsize=14, fontweight='bold', y=0.99)

plot_info = [
    ('test_acc',       'Test Accuracy',                axes[0, 0], 'Accuracy'),
    ('overfit_gap',    'Train-Test Gap (Overfitting)',  axes[0, 1], 'Gap'),
    ('avg_weight_mag', 'Avg Weight Magnitude',          axes[0, 2], 'Magnitude'),
    ('train_loss',     'Train Loss',                    axes[1, 0], 'Loss'),
    ('dormant_after',  'Dormant Units (Next Task)',     axes[1, 1], 'Proportion'),
    ('stable_rank',    'Stable Rank (Next Task)',       axes[1, 2], 'Stable Rank'),
]

for key, title, ax, ylabel in plot_info:
    for method, data in all_results.items():
        if key not in data or not data[key]:
            continue
        s = METHOD_STYLES.get(method, {'color': 'gray', 'ls': '-', 'lw': 1, 'label': method})
        y = np.array(data[key], dtype=float)
        x = np.arange(len(y))
        ax.plot(x, y, color=s['color'], ls=s['ls'], lw=s['lw'],
                label=s['label'], alpha=0.85)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Epoch', fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.grid(True, alpha=0.25, linewidth=0.5)
    ax.legend(fontsize=6.5, loc='best', framealpha=0.7, edgecolor='none')
    for tb in range(200, 4001, 200):
        ax.axvline(x=tb, color='gray', ls=':', alpha=0.25, lw=0.7)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig(os.path.join(results_dir, "anti_overfit_comparison.png"),
            dpi=200, bbox_inches='tight')
plt.show()
print(f"✓ Saved to {results_dir}/anti_overfit_comparison.png")

# ── Summary table ──
print(f"\n{'='*120}")
print(f"{'Method':<28} {'Optimizer':>10} {'TestAcc↑':>9} {'OvfGap↓':>9} "
      f"{'Dormant↓':>9} {'StbRank↑':>9} {'AvgW':>9} {'Time/ep':>9}")
print(f"{'='*120}")

for method, data in all_results.items():
    s = METHOD_STYLES.get(method, {})
    n = min(50, len(data['test_acc']))
    ta = np.mean(data['test_acc'][-n:])
    og = np.mean(data['overfit_gap'][-n:])
    da = np.nanmean(data['dormant_after'][-n:])
    sr = np.nanmean(data['stable_rank'][-n:])
    wm = np.mean(data['avg_weight_mag'][-n:])
    et = np.mean(data['epoch_time'])
    lbl = s.get('label', method)
    opt = CONFIGS[method]['optimizer']
    print(f"{lbl:<28} {opt:>10} {ta:>9.4f} {og:>9.4f} {da:>9.4f} "
          f"{sr:>9.1f} {wm:>9.4f} {et:>8.1f}s")

print(f"{'='*120}")